# 05 · The Two Clinical Truth Sets: ClinVar and CFTR2
### How to judge a variant predictor without fooling yourself

A variant-effect predictor (AlphaMissense, REVEL, EVE, CADD, SpliceAI…) gives you a number. But a number on its own is meaningless — *is 0.8 good?* You only find out whether a predictor is any good by comparing its calls against a **truth set**: a collection of variants whose real clinical status somebody already established.

For CFTR there are **two** truth sets you will meet again and again:

| Truth set | What it is | Kind of evidence |
|---|---|---|
| **ClinVar** | Public archive of clinical assertions from many labs | Prior clinical interpretation |
| **CFTR2** | CF-specific reference (cftr2.org) | Patient data **+ in-vitro function assays** |

This notebook is about what each one *actually is*, where each one can **mislead** you, and how to use them together. Two honesty notes up front:

- The **ClinVar** data here is **REAL** (~2,800 CFTR variants; every row is tagged `source == 'REAL'`).
- The **CFTR2** data here is a small **DEMO** set (a handful of curated variants embedded in the toolkit, tagged `source == 'DEMO'`). The real cftr2.org database is larger and behind a data-use agreement.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import toolkit as tk
import pandas as pd, numpy as np
%matplotlib inline

## 1 · ClinVar — what is it?

[ClinVar](https://www.ncbi.nlm.nih.gov/clinvar/) is a free, public archive run by the NIH. Clinical genetics labs around the world **submit their interpretation** of a variant: is it disease-causing or not? ClinVar aggregates those submissions and reports a clinical significance, usually one of:

- **Pathogenic** / **Likely pathogenic**
- **Uncertain significance** (a *VUS* — Variant of Uncertain Significance)
- **Likely benign** / **Benign**
- **Conflicting classifications** (labs disagreed)

Think of it as a giant, crowd-sourced ledger of *clinical opinion*. Its great strength is **breadth** — thousands of CFTR variants. Its weakness is that it is **heterogeneous**: different submitters, different rigor, different years. Let's load it.

In [2]:
clinvar = tk.load_clinvar()
print('rows:', len(clinvar))
print('source values:', clinvar['source'].unique(), ' <- REAL data')
clinvar.head()

rows: 2801
source values: ['REAL']  <- REAL data


,protein_variant,clinvar_sig,review_status,clinvar_call,source
0,Q493*,Pathogenic,reviewed by expert panel,pathogenic,REAL
1,D110H,Pathogenic; drug response,reviewed by expert panel,pathogenic,REAL
2,R117H,Pathogenic,practice guideline,pathogenic,REAL
3,R347P,Pathogenic,practice guideline,pathogenic,REAL
4,A455E,Pathogenic,practice guideline,pathogenic,REAL


Every row is `source == 'REAL'`: these are genuine ClinVar assertions for *CFTR*, not something the toolkit made up. Each row has:
- `protein_variant` — the amino-acid change (e.g. `G551D`)
- `clinvar_sig` — the raw, free-text clinical significance
- `review_status` — **how much confidence** to place in it (next section)
- `clinvar_call` — a tidy 3-class collapse of `clinvar_sig`

## 2 · Review status — the gold stars (⭐) that matter most

**This is the single most important slide in the notebook.** Not every ClinVar assertion deserves equal trust. ClinVar attaches a **review status**, which maps to a **0–4 gold-star** rating:

| Stars | Review status | Roughly means |
|:--:|---|---|
| 0 ⭐ | *no assertion criteria provided* | someone's opinion, no stated method |
| 1 ⭐ | *criteria provided, single submitter* | one lab, with a method |
| 1 ⭐ | *criteria provided, conflicting classifications* | labs disagree |
| 2 ⭐ | *criteria provided, multiple submitters, no conflicts* | several labs agree |
| 3 ⭐ | *reviewed by expert panel* | e.g. the CFTR2 / ClinGen expert panel |
| 4 ⭐ | *practice guideline* | encoded in clinical practice guidelines |

Let's count how our CFTR variants are distributed across these levels.

In [3]:
clinvar['review_status'].value_counts()

review_status
criteria provided, single submitter                     1373
criteria provided, multiple submitters, no conflicts     788
criteria provided, conflicting classifications           229
reviewed by expert panel                                 196
no assertion criteria provided                           145
no classification provided                                56
practice guideline                                        12
-                                                          2
Name: count, dtype: int64

**Read this carefully.** The largest bucket is *single submitter* (1 star) — one lab's call. Only a couple hundred are *expert panel* (3 stars), and a dozen are *practice guideline* (4 stars).

> **KEY LESSON:** a **1-star VUS** and a **3-star expert-panel Pathogenic** are *not* the same quality of evidence, even though both are 'in ClinVar'. When you build a benchmark, you often **filter to ≥2 stars** so a predictor is judged against assertions people actually trust — not against one lab's unreviewed guess. Throwing every star level into the same pile is one of the most common ways people accidentally lie to themselves with ClinVar.

## 3 · The tidy 3-class call — and why VUS are the whole point

The raw `clinvar_sig` text is messy (`'Pathogenic/Likely pathogenic'`, `'Conflicting classifications of pathogenicity'`, …). The toolkit collapses it into three clean buckets in `clinvar_call`. Let's count them.

In [4]:
counts = clinvar['clinvar_call'].value_counts()
print(counts)
n_vus = int(counts.get('uncertain', 0))
print(f'\nUncertain / VUS: {n_vus}  ({100*n_vus/len(clinvar):.0f}% of all rows)')

clinvar_call
uncertain     2246
pathogenic     540
benign          15
Name: count, dtype: int64

Uncertain / VUS: 2246  (80% of all rows)


Notice how **most CFTR variants in ClinVar are `uncertain` (VUS)**. That is not a bug — it is the *reason predictors exist*. If every variant already had a confident Pathogenic/Benign label, we would not need EVE or AlphaMissense. The huge pile of VUS is precisely the set of variants a lab gets back from sequencing and *cannot act on*. A good predictor's job is to help **re-classify** those VUS — so they are the interesting, unsolved cases, not the easy ones.

## 4 · How `cv_class` collapses the free text (and one deliberate choice)

`tk.cv_class()` is the little function that turns messy significance text into `pathogenic` / `benign` / `uncertain`. Let's watch it work on a few example strings.

In [5]:
examples = [
    'Pathogenic',
    'Benign/Likely benign',
    'Uncertain significance',
    'Conflicting classifications of pathogenicity',
]
pd.DataFrame({
    'input_significance': examples,
    'cv_class_output':    [tk.cv_class(s) for s in examples],
})

,input_significance,cv_class_output
0,Pathogenic,pathogenic
1,Benign/Likely benign,benign
2,Uncertain significance,uncertain
3,Conflicting classifications of pathogenicity,uncertain


Three of these are obvious. The fourth is the interesting one:

> **`Conflicting classifications` → `uncertain`.**

When labs *disagree* about a variant, the toolkit deliberately does **not** guess a winner — it treats the variant as `uncertain`. Why is that the *conservative* and correct choice for benchmarking?

- If we forced a conflicting variant into `pathogenic` or `benign`, we would be inventing a ground-truth label that **doesn't exist** — the experts themselves couldn't agree.
- Scoring a predictor against a made-up label would make the benchmark look cleaner than reality and could unfairly reward *or* punish the tool.

Collapsing conflicts to `uncertain` keeps them out of the Pathogenic-vs-Benign scoring, which is exactly where you want honest, high-confidence labels.

## 5 · CFTR2 — a disease-specific, *functional* reference

[CFTR2](https://cftr2.org) is different in kind from ClinVar. It is a **CF-specific** database that classifies *CFTR* variants as:

- **CF-causing**
- **CF-causing (mild / varying clinical consequence)**
- **Not CF-causing**
- *(unlisted)* — not yet enough evidence

Crucially, CFTR2's calls are built from **two kinds of evidence together**:
1. **Patient data** — real clinical outcomes across thousands of people with CF who carry the variant.
2. **In-vitro CFTR function assays** — measuring, in the lab, how much working chloride-channel the variant protein actually produces.

That second ingredient — *measured protein function* — is what makes CFTR2 special. Let's load the DEMO set the toolkit ships.

In [6]:
cftr2 = tk.load_cftr2_demo()
print('source values:', cftr2['source'].unique(), ' <- DEMO (small curated set)')
cftr2

source values: ['DEMO']  <- DEMO (small curated set)


,protein_variant,cftr2_class,source
0,G551D,CF-causing,DEMO
1,F508del,CF-causing,DEMO
2,R117H,CF-causing (mild),DEMO
3,R334W,CF-causing,DEMO
4,G85E,CF-causing,DEMO
5,D1152H,VUS,DEMO
6,R668C,Not CF-causing,DEMO
7,Tyr161Cys,VUS,DEMO
8,Gly970Asp,VUS,DEMO
9,Ser912Leu,VUS,DEMO


> ⚠️ **DEMO reminder:** every row is `source == 'DEMO'`. This is a tiny embedded teaching set, *not* the full cftr2.org database (which is larger and requires accepting a data-use agreement). The `cftr2_class` labels for the well-known alleles (G551D, F508del, R117H…) are genuine; the point of the DEMO set is to illustrate the *shape* of a CFTR2 join, not to be complete.

## 6 · ★ Key teaching point: why CFTR2 is a *more orthogonal* truth set

Here is the subtle trap that this whole notebook is building toward.

Many predictors — especially **supervised** ones like **REVEL** — were *trained* using variant labels that ultimately trace back to clinical databases in the **ClinVar lineage**. So if you 'benchmark' REVEL against ClinVar, you may partly be testing it **on data related to what it learned from**. It can look great for a circular reason: it's grading its own homework. This is called **circularity** (or label leakage).

**CFTR2 breaks that loop.** Because its calls incorporate **functional assay** evidence — a wet-lab measurement of channel function that a sequence model *never saw during training* — CFTR2 is a more **orthogonal** (independent) yardstick for judging sequence predictors. A predictor that agrees with CFTR2 is agreeing with *biology measured in a dish*, not merely with prior clinical opinion.

> **Rule of thumb:** use ClinVar for **breadth**, but lean on CFTR2 (and other functional data) when you need an **independent** check — particularly for supervised tools. We put this into practice in **notebook 08**, where we benchmark the predictors against both truth sets and watch which tools hold up when the yardstick is orthogonal.

## 7 · Where CFTR2 and ClinVar agree — and where they don't

Let's join the DEMO CFTR2 classes to the **REAL** ClinVar calls on `protein_variant`, for the handful of variants that appear in both. This is exactly the kind of small, careful cross-check you do before trusting a benchmark.

In [7]:
compare = cftr2.merge(
    clinvar[['protein_variant', 'clinvar_call', 'review_status']],
    on='protein_variant', how='inner',
)
compare = compare.rename(columns={'source': 'cftr2_source'})
compare

,protein_variant,cftr2_class,cftr2_source,clinvar_call,review_status
0,G551D,CF-causing,DEMO,pathogenic,practice guideline
1,R117H,CF-causing (mild),DEMO,pathogenic,practice guideline
2,R334W,CF-causing,DEMO,pathogenic,practice guideline
3,G85E,CF-causing,DEMO,pathogenic,practice guideline
4,D1152H,VUS,DEMO,uncertain,reviewed by expert panel
5,R668C,Not CF-causing,DEMO,uncertain,"criteria provided, conflicting classifications"


In [8]:
# Do the two truth sets point the same way?
def same_direction(row):
    cf = row['cftr2_class']
    cv = row['clinvar_call']
    if cf.startswith('CF-causing'):
        return 'agree' if cv == 'pathogenic' else 'DIFFER'
    if cf == 'Not CF-causing':
        return 'agree' if cv == 'benign' else 'DIFFER'
    return 'CFTR2 VUS / n.a.'   # CFTR2 itself is unsure

compare['verdict'] = compare.apply(same_direction, axis=1)
compare[['protein_variant', 'cftr2_class', 'clinvar_call',
         'review_status', 'verdict']]

,protein_variant,cftr2_class,clinvar_call,review_status,verdict
0,G551D,CF-causing,pathogenic,practice guideline,agree
1,R117H,CF-causing (mild),pathogenic,practice guideline,agree
2,R334W,CF-causing,pathogenic,practice guideline,agree
3,G85E,CF-causing,pathogenic,practice guideline,agree
4,D1152H,VUS,uncertain,reviewed by expert panel,CFTR2 VUS / n.a.
5,R668C,Not CF-causing,uncertain,"criteria provided, conflicting classifications",DIFFER


Read the `verdict` column:

- The classic CF alleles (**G551D, R334W, G85E, R117H**) are **CF-causing** in CFTR2 *and* **pathogenic** in ClinVar — the truth sets **agree**, which is reassuring.
- **R668C** is a great teaching case: CFTR2 calls it **Not CF-causing** (backed by function + patient data), while ClinVar sits at **uncertain**. Neither is 'wrong' — CFTR2 simply had *functional evidence* that let it reach a confident call where the crowd-sourced ClinVar view had not yet converged. That is orthogonality in action.

The lesson: when two truth sets **differ**, don't panic and don't average them — **ask what evidence each is built on**. A functional 'Not CF-causing' and a clinical 'uncertain' are answering slightly different questions.

## 8 · Key takeaways

1. **A predictor is only as trustworthy as the truth set you judge it against.** Pick the truth set deliberately.
2. **ClinVar is REAL, free, and broad** (~2,800 CFTR variants here) but **heterogeneous**. Always mind the **stars** — a 1-star VUS ≠ a 3-star expert-panel Pathogenic — and consider filtering to ≥2 stars.
3. **Most ClinVar CFTR variants are VUS.** That pile of uncertain variants is the *reason* predictors are useful, not noise to discard.
4. **`Conflicting` collapses to `uncertain` on purpose** — inventing a label the experts couldn't agree on would corrupt the benchmark.
5. **CFTR2 adds functional-assay evidence**, making it a more **orthogonal** yardstick — especially for **supervised** tools (REVEL) that may have learned from ClinVar-lineage labels. It's the antidote to **circularity**.
6. When the two truth sets **disagree**, look at *what evidence each carries* rather than forcing them to match.

➡️ **Next:** notebook 08 uses both of these truth sets to actually score the predictors — and shows why an orthogonal benchmark changes the leaderboard.